# nb_bronze_generic — metadata-driven Bronze ingestion

Reads `Files/config/sources.yaml` from the default lakehouse (**lh_bronze**) and ingests every source with
`enabled: true` and `loader: generic`:

1. builds the URL (placeholders `{start_date}`, `{end_date}`),
2. downloads the response and stores the raw file in `Files/raw/<name>/`,
3. parses it according to `format` and writes Delta table `<target_table>` with lineage columns
   `_source`, `_source_url`, `_ingested_at`.

**Adding a new source = adding an entry to `sources.yaml`.** No code change is needed for the supported formats
(`csv`, `json_records`, `json_columns`, `parquet`). See `docs/extending.md`.

In [1]:
config_path = "/lakehouse/default/Files/config/sources.yaml"
start_date = ""
end_date = ""
only_source = ""

StatementMeta(, c4b63827-fdb9-4ff7-941a-677bbad4d262, 3, Finished, Available, Finished, False)

In [2]:
import io
import json
import os
import re
from datetime import datetime, timezone
import pandas as pd
import requests
import yaml
from pyspark.sql import functions as F

with open(config_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)
defaults = config.get("defaults", {})
run_params = {
    "start_date": start_date or defaults.get("start_date"),
    "end_date": end_date or defaults.get("end_date"),
}
raw_root = "/lakehouse/default/" + defaults.get("raw_root", "Files/raw")
generic_sources = [
    s for s in config.get("sources", [])
    if s.get("enabled", False)
    and s.get("loader") == "generic"
    and (not only_source or s["name"] == only_source)
]
print(f"Run parameters: {run_params}")
print(f"Generic sources to ingest: {[s['name'] for s in generic_sources]}")

StatementMeta(, c4b63827-fdb9-4ff7-941a-677bbad4d262, 4, Finished, Available, Finished, False)

Run parameters: {'start_date': '2024-01-01', 'end_date': '2024-03-31'}
Generic sources to ingest: ['weather']


In [3]:
SUPPORTED_FORMATS = {"csv", "json_records", "json_columns", "parquet"}
FILE_EXTENSIONS = {"csv": "csv", "json_records": "json", "json_columns": "json", "parquet": "parquet"}

def clean_column(name: str) -> str:
    name = re.sub(r"[^0-9a-zA-Z]+", "_", str(name)).strip("_").lower()
    return name or "col"

def get_by_path(data, path):
    if not path:
        return data
    for key in path.split("."):
        data = data[int(key)] if isinstance(data, list) else data[key]
    return data

def parse(content: bytes, source: dict) -> pd.DataFrame:
    fmt = source["format"]
    if fmt == "csv":
        return pd.read_csv(io.BytesIO(content), sep=source.get("separator", ","))
    if fmt == "parquet":
        return pd.read_parquet(io.BytesIO(content))
    data = get_by_path(json.loads(content), source.get("record_path"))
    if fmt == "json_records":
        return pd.json_normalize(data)
    if fmt == "json_columns":
        return pd.DataFrame(data)
    raise ValueError(f"Unsupported format: {fmt}")

def ingest(source: dict) -> int:
    name = source["name"]
    fmt = source.get("format")
    if fmt not in SUPPORTED_FORMATS:
        raise ValueError(f"[{name}] format '{fmt}' is not supported by the generic loader")
    url = source["url"].strip().format(**run_params)
    headers = source.get("headers", {})
    response = requests.get(url, headers=headers, timeout=120)
    response.raise_for_status()
    ingested_at = datetime.now(timezone.utc)
    raw_dir = os.path.join(raw_root, name)
    os.makedirs(raw_dir, exist_ok=True)
    raw_file = os.path.join(raw_dir, f"{name}_{ingested_at:%Y%m%dT%H%M%S}.{FILE_EXTENSIONS[fmt]}")
    with open(raw_file, "wb") as f:
        f.write(response.content)
    pdf = parse(response.content, source)
    pdf.columns = [clean_column(c) for c in pdf.columns]
    df = (
        spark.createDataFrame(pdf.astype(str) if source.get("all_strings") else pdf)
        .withColumn("_source", F.lit(name))
        .withColumn("_source_url", F.lit(url))
        .withColumn("_ingested_at", F.lit(ingested_at.isoformat()).cast("timestamp"))
    )
    table = source.get("target_table", f"bronze_{name}")
    (df.write.format("delta")
       .mode(source.get("write_mode", "overwrite"))
       .option("overwriteSchema", "true")
       .saveAsTable(table))
    rows = df.count()
    print(f"[{name}] {rows} rows -> {table}; raw file: {raw_file}")
    return rows

StatementMeta(, c4b63827-fdb9-4ff7-941a-677bbad4d262, 5, Finished, Available, Finished, False)

In [4]:
results, errors = {}, {}
for source in generic_sources:
    try:
        results[source["name"]] = ingest(source)
    except Exception as exc:
        errors[source["name"]] = str(exc)
        print(f"[{source['name']}] FAILED: {exc}")
print("Loaded:", results)
if errors:
    raise RuntimeError(f"Generic ingestion failed for: {errors}")

StatementMeta(, c4b63827-fdb9-4ff7-941a-677bbad4d262, 6, Finished, Available, Finished, False)

[weather] 91 rows -> bronze_weather; raw file: /lakehouse/default/Files/raw/weather/weather_20261125T095703.json
Loaded: {'weather': 91}
